# Greenwashing Pipeline — Batch Runner & Visualiser

This notebook executes the NLP pipeline (`src.pilot`) across **all PDF reports** in `inputs/`,
aggregates the per-company JSON outputs into a single dataset, and visualises the resulting
Greenwashing (GW) scores.

**Workflow:**
1. **Cell 1** — run `src.pilot` for every `inputs/<company>-<year>.pdf` via subprocess; collect stdout JSON.
2. **Cell 2** — assemble results into a Pandas DataFrame and display the summary table.
3. **Cell 3** — produce two publication-quality plots: GW trend lines and a VUI vs SPI scatter.
4. **Cell 4** — regression check: compare DWS batch results against the stored `outputs/run_*.json` baselines.

**Before running:** set `ENABLE_NLI = False` for a fast parse-only pass (no model download); set `True` for full thesis-quality scores (downloads `facebook/bart-large-mnli` ~1.6 GB on first use).

Run from the **repo root**: `jupyter lab scripts/run_all.ipynb`

In [ ]:
import os
import glob
import subprocess
import json
import re

# ---------------------------------------------------------------------------
# Known CI emission time-series (tCO2e).
# Format: company_name_lowercase -> "year:value,year:value,..."
# Add more companies here as data becomes available.
# ---------------------------------------------------------------------------
CI_VALUES = {
    "dws": "2022:2694,2023:2126,2024:1497",
}

# Toggle NLI semantic scoring.
# False = fast rule-only run (no model download, ~1-3 min for all PDFs).
# True  = full NLI scoring with bart-large-mnli (~1.6 GB, slow on first use).
ENABLE_NLI = True

# ---------------------------------------------------------------------------

results = []
pdf_paths = sorted(glob.glob("inputs/*.pdf"))
print(f"Found {len(pdf_paths)} PDFs.\n")

for pdf_path in pdf_paths:
    stem = os.path.splitext(os.path.basename(pdf_path))[0]  # e.g. "dws-2024"
    m = re.match(r"^(.+)-(\d{4})$", stem)
    if not m:
        print(f"  Skipping (unexpected filename format): {pdf_path}")
        continue
    company, year = m.group(1).lower(), int(m.group(2))

    outdir = f"outputs/batch/{company}"
    os.makedirs(outdir, exist_ok=True)

    cmd = [
        "python3", "-m", "src.pilot",
        "--pdf",    pdf_path,
        "--year",   str(year),
        "--outdir", outdir,
    ]
    if ENABLE_NLI:
        cmd.append("--enable-nli")
    if company in CI_VALUES:
        cmd += ["--ci-values", CI_VALUES[company]]

    print(f"  {company:<15} {year} ...", end=" ", flush=True)
    try:
        proc = subprocess.run(
            cmd, capture_output=True, text=True, timeout=600
        )
        if proc.returncode != 0:
            print("ERROR")
            results.append({
                "company": company, "year": year,
                "error": proc.stderr.strip()[-300:],
            })
        else:
            data = json.loads(proc.stdout)
            print(f"GW={data['gw']:.3f}  ({data['band']})")
            results.append({
                "company":           company,
                "year":              year,
                "vui_norm":          data["vui_norm"],
                "spi_hybrid":        data["spi_hybrid"],
                "gw":                data["gw"],
                "band":              data["band"],
                "ci_value":          data.get("ci_value"),
                "included_sections": data.get("included_sections", []),
            })
    except subprocess.TimeoutExpired:
        print("TIMEOUT")
        results.append({"company": company, "year": year, "error": "TIMEOUT"})
    except Exception as exc:
        print(f"EXCEPTION: {exc}")
        results.append({"company": company, "year": year, "error": str(exc)})

ok  = sum(1 for r in results if "gw"    in r)
err = sum(1 for r in results if "error" in r)
print(f"\nFinished: {ok} succeeded, {err} failed.")

In [ ]:
import pandas as pd

success_rows = [r for r in results if "gw"    in r]
error_rows   = [r for r in results if "error" in r]

if error_rows:
    print("Errors:")
    for e in error_rows:
        print(f"  {e['company']} {e['year']}: {e['error'][:120]}")
    print()

df = (
    pd.DataFrame(success_rows)
    .sort_values(["company", "year"])
    .reset_index(drop=True)
)

df[["company", "year", "vui_norm", "spi_hybrid", "gw", "band", "ci_value"]]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

BAND_COLORS = {"low": "#2ecc71", "medium": "#f39c12", "high": "#e74c3c"}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "Greenwashing Pipeline — Company Overview",
    fontsize=14, fontweight="bold",
)

# --- Plot 1: GW trend lines per company ---
sns.lineplot(
    data=df, x="year", y="gw", hue="company",
    marker="o", linewidth=1.5, ax=axes[0],
)
axes[0].axhline(0.30, color="gray", linestyle="--", linewidth=0.8, alpha=0.6,
                label="low / medium boundary")
axes[0].axhline(0.60, color="gray", linestyle=":",  linewidth=0.8, alpha=0.6,
                label="medium / high boundary")
axes[0].set_title("GW Score Trend by Company")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("GW Score")
axes[0].set_ylim(0, 1.05)
axes[0].legend(fontsize=7, ncol=2, title="Company")

# --- Plot 2: VUI vs SPI scatter, coloured by GW band ---
for band, colour in BAND_COLORS.items():
    subset = df[df["band"] == band]
    axes[1].scatter(
        subset["vui_norm"], subset["spi_hybrid"],
        c=colour, label=band.capitalize(),
        alpha=0.8, s=60, edgecolors="white", linewidths=0.5,
    )
for _, row in df.iterrows():
    axes[1].annotate(
        f"{row['company'][:4]}\n'{str(row['year'])[2:]}",
        (row["vui_norm"], row["spi_hybrid"]),
        fontsize=6, ha="center", va="bottom", alpha=0.7,
    )
axes[1].set_title("VUI vs SPI Hybrid — Coloured by GW Band")
axes[1].set_xlabel("VUI  (vague language, normalised)")
axes[1].set_ylabel("SPI Hybrid  (specificity)")
axes[1].legend(title="Band")

plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
plt.savefig("outputs/gw_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/gw_overview.png")

In [ ]:
import glob as _glob

# ---------------------------------------------------------------------------
# Regression check: compare batch DWS results against stored baselines.
# Stored baselines live in outputs/run_<year>.json (DWS-only runs).
# Expected behaviour:
#   2021 / 2022 — vui_norm, spi_hybrid, gw should match within TOL
#   2023        — gw will differ (CI now applied: ci_value=1.0 lowers GW)
#   2024        — should match (both runs used ci-values)
# ---------------------------------------------------------------------------
BASELINE_DIR = "outputs"
TOL = 0.001  # acceptable floating-point delta

baselines = {}
for path in sorted(_glob.glob(f"{BASELINE_DIR}/run_*.json")):
    with open(path) as f:
        b = json.load(f)
    baselines[b["year"]] = b

if not baselines:
    print("No baseline run_*.json files found in outputs/ — skipping validation.")
else:
    dws_batch = df[df["company"] == "dws"].set_index("year")

    print(f"{'Year':>4}  {'Metric':<12}  {'Baseline':>10}  {'Batch':>10}  Status")
    print("-" * 58)
    for year, b in sorted(baselines.items()):
        if year not in dws_batch.index:
            print(f"{year:>4}  (not in batch results — was this PDF skipped?)")
            continue
        row = dws_batch.loc[year]
        checks = {
            "vui_norm":   (round(b["vui"]["vui_norm"], 4), row["vui_norm"]),
            "spi_hybrid": (round(b["spi_hybrid"],       4), row["spi_hybrid"]),
            "gw":         (round(b["gw"]["gw"],         4), row["gw"]),
        }
        for metric, (base_val, batch_val) in checks.items():
            match = abs(base_val - batch_val) <= TOL
            flag  = "OK" if match else "DIFF  (expected if CI now applied)"
            print(
                f"{year:>4}  {metric:<12}  {base_val:>10.4f}  "
                f"{batch_val:>10.4f}  {flag}"
            )
        print()